In [1]:
from ech_discovery.notebook import *
import re


# Load data

In [2]:
bool_cols = [
    Strings.esni_ech_connection,
    Strings.detection_without_dns,
]

In [3]:
df = (
    pl.read_csv(
        config.INTERIM_DATA_DIR / "2026-05-28-related-work-ech.tsv", separator="\t"
    )
    .drop("")
    .rename(
        {
            C.citekey: Strings.paper,
            C.esni_ech_connection: Strings.esni_ech_connection,
            C.detection_without_dns: Strings.detection_without_dns,
        }
    )
    .with_columns(
        pl.col(bool_cols)
        .replace_strict(
            {
                None: False,
                "x": True,
                "-": False,
            },
            return_dtype=pl.Boolean,
        )
        .fill_null(False)
    )
)

In [4]:
def wrap_column_with_citeauthor(latex: str, column_values: list[str]) -> str:
    """Replace each value in the column with \\citeauthor{value}."""
    for val in column_values:
        # Escape val for regex, then replace bare occurrences in the table body
        if val == Strings.this_work_citekey:
            continue
        escaped = re.escape(val)
        latex = re.sub(
            rf"(?<![{{\\]){escaped}(?![}}])",  # not already wrapped
            rf"\\citeauthor{{{val}}}~\\cite{{{val}}}",
            latex,
        )
    return latex

In [5]:
def build_latex(
    df: pl.DataFrame,
    title: str,
    group_by_kind: bool = False,
) -> str:
    TICK = "XCHECKX"
    XMARK = "XMARKX"
    BOOL_COLS = list(
        filter(
            lambda x: x in bool_cols,
            df.columns,
        )
    )
    df_gt = df.with_columns(
        pl.when(pl.col(c)).then(pl.lit(TICK)).otherwise(pl.lit(XMARK)).alias(c)
        for c in BOOL_COLS
    )
    latex = (
        GT(df_gt)
        .cols_align("center", BOOL_COLS)
        .tab_header(title=title)
        .fmt(lambda x: f"'{str(x)[-2:]}", columns=Strings.year)
        .as_latex()
    )
    for k, v in {
        TICK: Strings.latex_cmark,
        XMARK: Strings.latex_xmark,

    }.items():
        latex = latex.replace(k, v)

    lines = latex.split("\n")
    for k, v in {
        **{f"{col}": rf"\rotatebox[origin=lb]{{45}}{{{col}}}" for col in BOOL_COLS},
    }.items():
        lines[:18] = [line.replace(k, v) for line in lines[:18]]

    latex = "\n".join(lines)
    latex = wrap_column_with_citeauthor(latex, df[Strings.paper].to_list())

    return latex


title = """
Overview of studies measuring the deployment of related work. 
"""
t = build_latex(
    df.rename({}).sort(Strings.year),
    title=title,
)
print(ndpi.visualization.tables.fix_gt_table(t, "tab:deployment-measurements"))

\begin{table*}[!t]
\caption{Overview of studies measuring the deployment of related work. 
}
\label{tab:deployment-measurements} 
\begin{tabular}{rllccl}
\toprule
Year & Paper & Measurement & \rotatebox[origin=lb]{45}{ESNI/ECH Connection} & \rotatebox[origin=lb]{45}{Detection without DNS} & Limitations \\ 
\midrule\addlinespace[2.5pt]
'19 & \citeauthor{cgh-ieetc-19}~\cite{cgh-ieetc-19} & ESNI & \cmark & \xmark & Only domains with ECH configuration in DNS considered \\
'22 & \citeauthor{hpg-madne-22}~\cite{hpg-madne-22} & ESNI & \cmark & \xmark & Only a small list of censored domains consisdered \\
'22 & \citeauthor{tkk-matec-22}~\cite{tkk-matec-22} & ESNI, ECH & \cmark & \xmark & Only domains with ECH configuration in DNS considered \\
'23 & \citeauthor{zsc-flasa-23}~\cite{zsc-flasa-23} & ECH & \xmark & \xmark & Only analyze DNS records \\
'24 & \citeauthor{dhzyl-eedhr-24}~\cite{dhzyl-eedhr-24} & ECH & \xmark & \xmark & Active connection only to own test-instance \\
'25 & \citeauthor{m